## Goal

Generate a synthetic CSV file that simulates collected rural community demand data.

In [1]:
import sys
from pathlib import Path

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from agents.generate_demo_data import create_demo_dataset

df = create_demo_dataset(rows=250)

df.head()


,response_id,age_group,gender,region,occupation,source,main_problem,needed_service,service_category,importance_level,current_solution,satisfaction_level,would_pay,monthly_budget,urgency_level,frequency_of_problem,preferred_solution_type,opinion_text,date_collected
0,1,25-34,Prefer not to say,Al Madam,Employee,voice_note,Low visibility for local produce,Farm product marketplace,Agriculture,Low,Sell through relatives,Satisfied,Yes,31-50 AED,Very urgent,Monthly,Phone call,Small farms need a simple way to sell dates an...,2026-04-20
1,2,25-34,Female,Liwa,Retired,qr_survey,Students need exam preparation,Student tutoring,Education,High,Online videos,Very dissatisfied,Maybe,31-50 AED,Very urgent,Rarely,Mobile app,"Many students need affordable tutoring nearby,...",2026-06-02
2,3,35-44,Prefer not to say,Masfout,Teacher,interview,No pharmacy delivery,Pharmacy delivery,Delivery,Medium,Drive to city,Dissatisfied,Maybe,10-30 AED,Very urgent,Monthly,WhatsApp,We need faster delivery because shops are far ...,2026-05-22
3,4,45-54,Female,Ras Al Khaimah rural area,Student,shop_owner_note,No simple ordering system for farms,Weekly farm box service,Agriculture,Medium,Sell through relatives,Very dissatisfied,Maybe,100+ AED,Not urgent,Rarely,Website,People would buy more local produce if orderin...,2026-03-31
4,5,35-44,Female,Al Madam,Teacher,qr_survey,Difficult transport for appointments,Community shuttle,Transport,Medium,Use taxi,Neutral,Maybe,10-30 AED,Very urgent,Weekly,Website,Taxi services are expensive or not always avai...,2026-06-26


In [2]:
import pandas as pd
from pathlib import Path
import re


def get_next_response_number(existing_df):

    if existing_df is None or existing_df.empty:
        return 1

    if "response_id" not in existing_df.columns:
        return len(existing_df) + 1

    numbers = []

    for value in existing_df["response_id"].dropna():
        value = str(value).strip()

        match = re.match(r"R(\d+)$", value)

        if match:
            numbers.append(int(match.group(1)))

    if numbers:
        return max(numbers) + 1

    return len(existing_df) + 1


def assign_short_unique_ids(new_df, start_number):

    new_df = new_df.copy()

    if "batch_id" in new_df.columns:
        new_df = new_df.drop(columns=["batch_id"])

    new_df["response_id"] = [
        f"R{number:06d}"
        for number in range(start_number, start_number + len(new_df))
    ]

    return new_df


def append_to_raw_dataset(new_df, raw_output_path):

    raw_output_path = Path(raw_output_path)
    raw_output_path.parent.mkdir(parents=True, exist_ok=True)

    if raw_output_path.exists():
        old_df = pd.read_csv(raw_output_path)

        if "batch_id" in old_df.columns:
            old_df = old_df.drop(columns=["batch_id"])
    else:
        old_df = pd.DataFrame()

    next_number = get_next_response_number(old_df)

    new_df = assign_short_unique_ids(
        new_df=new_df,
        start_number=next_number
    )

    if not old_df.empty:
        combined_df = pd.concat([old_df, new_df], ignore_index=True)
    else:
        combined_df = new_df.copy()

    before_dedup = len(combined_df)

    combined_df = combined_df.drop_duplicates(
        subset=["response_id"],
        keep="last"
    )

    duplicates_removed = before_dedup - len(combined_df)

    combined_df.to_csv(raw_output_path, index=False)

    print("Raw dataset updated.")
    print("Saved to:", raw_output_path)
    print("New rows added:", len(new_df))
    print("First new response ID:", new_df["response_id"].iloc[0])
    print("Last new response ID:", new_df["response_id"].iloc[-1])
    print("Rows before duplicate removal:", before_dedup)
    print("Duplicates removed:", duplicates_removed)
    print("Final rows:", len(combined_df))

    return combined_df


output_path = PROJECT_ROOT / "data" / "raw" / "responses_raw.csv"

combined_df = append_to_raw_dataset(
    new_df=df,
    raw_output_path=output_path
)

print("Saved CSV to:", output_path)

Raw dataset updated.
Saved to: C:\Users\ASUS\PycharmProjects\PythonProject8\data\raw\responses_raw.csv
New rows added: 250
First new response ID: R000251
Last new response ID: R000500
Rows before duplicate removal: 400
Duplicates removed: 0
Final rows: 400
Saved CSV to: C:\Users\ASUS\PycharmProjects\PythonProject8\data\raw\responses_raw.csv


In [3]:
print("Shape:", df.shape)
print()
print("Columns:")
print(df.columns.tolist())
print()
print("Service categories:")
print(df["service_category"].value_counts())
print()
print("Sources:")
print(df["source"].value_counts())

Shape: (250, 19)

Columns:
['response_id', 'age_group', 'gender', 'region', 'occupation', 'source', 'main_problem', 'needed_service', 'service_category', 'importance_level', 'current_solution', 'satisfaction_level', 'would_pay', 'monthly_budget', 'urgency_level', 'frequency_of_problem', 'preferred_solution_type', 'opinion_text', 'date_collected']

Service categories:
service_category
Transport      50
Tourism        45
Agriculture    41
Repair         40
Education      37
Delivery       37
Name: count, dtype: int64

Sources:
source
qr_survey          56
shop_owner_note    56
interview          51
paper_form         45
voice_note         42
Name: count, dtype: int64


In [4]:
from agents.collector_agent import create_demo_dataset
df = create_demo_dataset(rows=100)
df.head()

Demo data generated. Rows: 100. Saved to C:\Users\ASUS\PycharmProjects\PythonProject8\data\raw\responses_raw.csv


,response_id,age_group,gender,region,occupation,source,main_problem,needed_service,service_category,importance_level,frequency_of_problem,preferred_solution_type,monthly_budget,opinion_text,date_collected,import_batch_id,imported_at,source_file_name
0,DEMO_R0001,25-34,male,Liwa,small grocery owner,government information,Transport to services in Al Ain is difficult f...,scheduled rural shuttle,Rural Transport,5,occasionally,WhatsApp bot,101-250 AED,"A scheduled shuttle to Al Ain for errands, cli...",2026-06-24,DEMO,2026-06-27T07:23:00,synthetic_al_quaa_demo
1,DEMO_R0002,25-34,prefer not to say,Al Ain outskirts,camel farm owner,survey,Camel farms in Al Qua'a struggle to get fast v...,same-day mobile camel vet booking,Veterinary & Camel Care,4,daily,SMS updates,251-500 AED,A WhatsApp booking service for camel vets woul...,2026-06-27,DEMO,2026-06-27T07:23:00,synthetic_al_quaa_demo
2,DEMO_R0003,35-44,male,Al Wathba,teacher,volunteer,"Clinic visits require long travel, especially ...",mobile health checkup visits,Mobile Healthcare,5,daily,phone call service,51-100 AED,Mobile checkup visits would help residents get...,2026-05-14,DEMO,2026-06-27T07:23:00,synthetic_al_quaa_demo
3,DEMO_R0004,45-54,male,Liwa,home-based food seller,survey,Irrigation and water pump issues take too long...,farm equipment maintenance coordination,Farm Operations,5,weekly,web dashboard,101-250 AED,A maintenance coordination service for irrigat...,2026-03-29,DEMO,2026-06-27T07:23:00,synthetic_al_quaa_demo
4,DEMO_R0005,18-24,male,Al Qua'a,camel farm worker,government information,Farmers cannot easily compare camel feed suppl...,camel feed and vet price comparison,Veterinary & Camel Care,4,weekly,web dashboard,0-50 AED,"A simple comparison tool for feed, vet visits,...",2026-05-13,DEMO,2026-06-27T07:23:00,synthetic_al_quaa_demo
